In [0]:
GOLD_PATH = "/Volumes/workspace/default/logs_volume/gold"
SILVER_PATH = "/Volumes/workspace/default/logs_volume/silver"

In [0]:
silver_df = spark.read.format("delta").load(SILVER_PATH)

print(f"Rows loaded: {silver_df.count():,}")

In [0]:
from pyspark.sql.functions import (
    col, count, sum, avg, max, min, when,
    hour, date_trunc, round,
    window, lag, rank
)

hourly_df = (silver_df
    .withColumn("hour_bucket", date_trunc("hour", col("timestamp")))
    .groupBy("hour_bucket")
    .agg(
        count("*").alias("total_requests"),
        sum("bytes").alias("total_bytes"),
        count(when(col("status") == 404, 1)).alias("errors_404"),
        count(when(col("status") == 500, 1)).alias("errors_500"),
        count(when(col("status").between(200, 299), 1)).alias("success_count")
    )
    .withColumn("error_rate",
        round((col("errors_404") + col("errors_500")) / col("total_requests"), 4)
    )
    .withColumn("success_rate",
        round(col("success_count") / col("total_requests"), 4)
    )
    .orderBy("hour_bucket")
)

hourly_df.show(10)

In [0]:
from pyspark.sql.window import Window
# Define the window: partition by nothing (whole dataset), order by time, 
# look back 2 rows (= current + 2 previous hours = 3hr window)
hourly_window = (Window
    .orderBy("hour_bucket")
    .rowsBetween(-2, 0))  # -2 = 2 rows back, 0 = current row

hourly_df = hourly_df.withColumn(
    "rolling_3hr_avg_requests",
    round(avg("total_requests").over(hourly_window), 2)
)

hourly_df.show(10)

In [0]:
from pyspark.sql.functions import countDistinct
ip_df = (silver_df
    .groupBy("host")
    .agg(
        count("*").alias("total_requests"),
        sum("bytes").alias("total_bytes_transferred"),
        count(when(col("status") >= 400, 1)).alias("total_errors"),
        round(
            count(when(col("status") >= 400, 1)) / count("*") * 100, 2
        ).alias("error_rate_pct"),
        min("timestamp").alias("first_seen"),
        max("timestamp").alias("last_seen"),
        countDistinct("endpoint").alias("unique_endpoints_hit")
    )
    .orderBy("total_requests", ascending=False)
)

ip_df.show(10)

In [0]:
ip_hourly_df = (silver_df
    .groupBy(
        "host",
        window(col("timestamp"), "1 hour").alias("time_window")
    )
    .agg(count("*").alias("requests_in_hour"))
    .select(
        col("host"),
        col("time_window.start").alias("hour_start"),
        col("requests_in_hour")
    )
    .orderBy("host", "hour_start")
)

ip_hourly_df.show(10)

In [0]:
endpoint_df = (silver_df
    .groupBy("endpoint")
    .agg(
        count("*").alias("total_hits"),
        round(avg("bytes"), 2).alias("avg_bytes"),
        max("bytes").alias("max_bytes"),
        count(when(col("status") == 404, 1)).alias("not_found_count"),
        count(when(col("status") == 200, 1)).alias("success_count"),
        countDistinct("host").alias("unique_visitors")
    )
    .orderBy("total_hits", ascending=False)
)

endpoint_df.show(20)

In [0]:
broken_endpoints = (endpoint_df
    .withColumn(
        "not_found_rate_pct",
        round(col("not_found_count") / col("total_hits") * 100, 2)
    )
    .filter(col("total_hits") > 50)   # ignore low-traffic endpoints
    .orderBy("not_found_rate_pct", ascending=False)
)

broken_endpoints.show(20)

In [0]:
# Hourly stats
(hourly_df
    .write.format("delta")
    .mode("overwrite")
    .save(f"{GOLD_PATH}/hourly_stats"))

# IP behaviour
(ip_df
    .write.format("delta")
    .mode("overwrite")
    .save(f"{GOLD_PATH}/ip_behaviour"))

# Endpoint stats
(endpoint_df
    .write.format("delta")
    .mode("overwrite")
    .save(f"{GOLD_PATH}/endpoint_stats"))

print("All 3 Gold tables written successfully.")

In [0]:
for table in ["hourly_stats", "ip_behaviour", "endpoint_stats"]:
    df = spark.read.format("delta").load(f"{GOLD_PATH}/{table}")
    print(f"{table}: {df.count():,} rows")